# 06 · Afinamiento de fast-plate-ocr con placas ecuatorianas

**SIRAP – Proyecto Integrador en IA (MIAR0545)** · Eduardo Alejandro Ceballos Jijón · Christian Xavier Valle Maridueña

El OCR de SIRAP es `fast-plate-ocr` con el modelo **`cct-s-v2-global-model`**, entrenado con placas de más de 65 países… pero **Ecuador no está en su lista de regiones**. En las placas reales de SAVH Guayaquil ya acierta el **93,7 %** (EMR, sección 5 del notebook de backend). Este notebook prueba si **seguir entrenándolo con placas ecuatorianas** mejora ese número, sin engañarnos con la evaluación:

1. **Datos:** recortes legibles de SAVH Guayaquil con la referencia corregida (sección 5 del backend) y, si existen, recortes de la garita hechos por YOLO11-s (sección 7).
2. **Separación por placa**, no por imagen: una misma placa nunca está a la vez en entrenamiento y en prueba (si no, el modelo «memoriza» la placa y el resultado se infla).
3. **Línea base:** el modelo original medido en el mismo *test*.
4. **Afinamiento** desde los pesos `.keras` publicados por el autor de la librería, con *learning rate* bajo y aumentos suaves.
5. **Comparación antes / después** en *test*: EMR, CER, intervalos de confianza y **prueba de McNemar** (con un *test* de pocas decenas de placas, una diferencia de 1–2 aciertos puede ser azar).
6. **Exportación a ONNX** y, solo si mejora de verdad, copia a Drive para que la app la use.
7. **Opcional (sección 7 del backend):** medir ambos OCR con los **recortes del detector** en las imágenes de la garita.

Requisitos: GPU (T4 alcanza; el modelo es pequeño), Drive con `MyDrive/proyecto_integrador_placas` y la referencia `ocr/savh_ground_truth_corregido.csv` que genera la sección 5 del notebook de backend.
Documentación: `docs/optimizacion.md` (sección 8) del repositorio.

## 0. Preparación

In [ ]:
!pip install -q "fast-plate-ocr[train,onnx]==1.1.0" "tensorflow[and-cuda]" Levenshtein
%env TF_CPP_MIN_LOG_LEVEL=3
%env KERAS_BACKEND=tensorflow

In [ ]:
import json, re, shutil, subprocess, sys, zipfile
from math import comb
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 0
WORK_DIR = Path("/content/proyecto_integrador_placas")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/proyecto_integrador_placas")
except ImportError:
    print("Fuera de Colab: se usa una carpeta local.")
OCR_DIR = WORK_DIR / "ocr"
FT_DIR = OCR_DIR / "afinamiento"            # todo lo de este notebook queda en Drive
FT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/ocr_ft")             # disco local: imágenes y entrenamiento (más rápido que Drive)
LOCAL.mkdir(parents=True, exist_ok=True)

# Normalización de SIRAP (la misma de la app y del notebook de backend), desde el repositorio público
if not Path("/content/SIRAP").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/eduardoceballos666/SIRAP", "/content/SIRAP"], check=True)
sys.path.insert(0, "/content/SIRAP")
from src.data_processing import clean_text, normalize_plate, plate_distance
print("Salidas en:", FT_DIR)

### Parámetros
`INCLUDE_UNCONFIRMED`: las 69 referencias de SAVH que todavía son sugerencias sin confirmar en la hoja de revisión. Por defecto **no** entran ni al entrenamiento ni a la prueba (una etiqueta equivocada enseña un error y además lo premia en la evaluación). Cuando la hoja esté confirmada, vuelva a correr este notebook con `True`.

In [ ]:
INCLUDE_UNCONFIRMED = False      # True cuando la hoja «Revisión placas SAVH» esté confirmada
SPLIT = {"train": 0.60, "val": 0.15, "test": 0.25}   # por PLACA, no por imagen
BASE_MODEL = "cct_s_v2_global"   # el modelo que usa la app (cct-s-v2-global-model)
EPOCHS, BATCH, LR, WEIGHT_DECAY = 40, 32, 3e-4, 5e-4
EARLY_STOPPING_PATIENCE = 10
OCR_CONF_MIN = 0.99              # umbral de la app (confianza = carácter menos seguro)

## 1. Datos: SAVH Guayaquil (y garita, si existe)

In [ ]:
SAVH_DIR = Path("/content/savh_ground_truth_subset")
SAVH_ZIP = next((p for p in [WORK_DIR / "savh_ground_truth_subset.zip", Path("/content/savh_ground_truth_subset.zip")] if p.exists()), None)
if not (SAVH_DIR / "ground_truth.csv").exists():
    assert SAVH_ZIP, "Falta savh_ground_truth_subset.zip en la carpeta del proyecto en Drive."
    with zipfile.ZipFile(SAVH_ZIP) as archive:
        archive.extractall(SAVH_DIR)

ref_path = OCR_DIR / "savh_ground_truth_corregido.csv"
assert ref_path.exists(), "Falta ocr/savh_ground_truth_corregido.csv: corra la sección 5 del notebook de backend."
ref = pd.read_csv(ref_path, dtype=str, keep_default_na=False)
ok_states = {"camara_formato_valido", "confirmada", "corregida_por_revisor"} | ({"sugerida_sin_confirmar"} if INCLUDE_UNCONFIRMED else set())
unique = ref["duplicado_de_fila"].isin(["", "<NA>", "nan"])
pool = ref[unique & ref["estado_gt"].isin(ok_states)].copy()
pool["plate_text"] = pool["plate_gt"].map(clean_text)                  # lo que está impreso: GBA1234 o GMS429
pool = pool[pool["plate_text"].str.fullmatch(r"[A-Z]{3}\d{3,4}")]       # solo formatos ecuatorianos válidos
pool["src_path"] = pool["image_path"].map(lambda p: str(SAVH_DIR / p))
pool["origen"] = "SAVH"
print("Estados en la referencia:"); display(ref.loc[unique, "estado_gt"].value_counts())
print(f"SAVH utilizables: {len(pool)} imágenes · {pool['plate_text'].nunique()} placas distintas")

# Opcional: recortes de la garita hechos por el detector (se arman en la sección 7 de este notebook)
GARITA_CROPS = FT_DIR / "garita_recortes" / "recortes.csv"
if GARITA_CROPS.exists():
    garita = pd.read_csv(GARITA_CROPS, dtype=str)
    garita = garita.assign(plate_text=garita["plate_gt"].map(clean_text), src_path=garita["crop_path"], origen="garita")
    pool = pd.concat([pool, garita[["plate_text", "src_path", "origen"]]], ignore_index=True)
    print(f"+ {len(garita)} recortes de la garita")
pool = pool[["plate_text", "src_path", "origen"]].reset_index(drop=True)

### Separación por placa
Se reparten las **placas distintas** (no las imágenes) en *train* / *val* / *test* con semilla fija. Así, si una placa aparece en varias imágenes, todas van al mismo grupo.

In [ ]:
rng = np.random.default_rng(SEED)
plates = np.array(sorted(pool["plate_text"].unique()))
rng.shuffle(plates)
n = len(plates)
cut1, cut2 = int(n * SPLIT["train"]), int(n * (SPLIT["train"] + SPLIT["val"]))
group = {p: "train" for p in plates[:cut1]} | {p: "val" for p in plates[cut1:cut2]} | {p: "test" for p in plates[cut2:]}
pool["split"] = pool["plate_text"].map(group)
assert not (set(pool.loc[pool.split == "train", "plate_text"]) & set(pool.loc[pool.split == "test", "plate_text"]))
display(pd.crosstab(pool["split"], pool["origen"], margins=True))

# Formato de fast-plate-ocr: CSV con image_path (relativo al CSV) y plate_text
DATA = LOCAL / "dataset"
for split in ("train", "val", "test"):
    (DATA / split).mkdir(parents=True, exist_ok=True)
rows = {s: [] for s in ("train", "val", "test")}
for i, r in pool.iterrows():
    dst = DATA / r.split / f"{i:05d}{Path(r.src_path).suffix.lower() or '.jpg'}"
    shutil.copy(r.src_path, dst)
    rows[r.split].append({"image_path": f"{r.split}/{dst.name}", "plate_text": r.plate_text})
for split, items in rows.items():
    pd.DataFrame(items).to_csv(DATA / f"{split}_annotations.csv", index=False)
pool.to_csv(FT_DIR / "particion.csv", index=False)
print("CSV listos en", DATA)

## 2. Modelo base y su configuración

In [ ]:
BASE_URL = "https://github.com/ankandrew/fast-plate-ocr/releases/download/arg-plates"
for name in (f"{BASE_MODEL}.keras", f"{BASE_MODEL}_model_config.yaml", f"{BASE_MODEL}_plate_config.yaml", f"{BASE_MODEL}.onnx"):
    if not (LOCAL / name).exists():
        subprocess.run(["wget", "-q", f"{BASE_URL}/{name}", "-O", str(LOCAL / name)], check=True)
BASE_KERAS, MODEL_CFG = LOCAL / f"{BASE_MODEL}.keras", LOCAL / f"{BASE_MODEL}_model_config.yaml"
PLATE_CFG, BASE_ONNX = LOCAL / f"{BASE_MODEL}_plate_config.yaml", LOCAL / f"{BASE_MODEL}.onnx"
for split in ("train", "val", "test"):
    !fast-plate-ocr validate-dataset --annotations-file {DATA}/{split}_annotations.csv --plate-config-file {PLATE_CFG}

## 3. Evaluación común
La misma medición para el modelo original y el afinado, sobre las mismas imágenes de *test*:
- **EMR**: placa exacta después de la normalización de SIRAP (`XXX-0000`), como en la app.
- **CER**: errores por carácter.
- **Aceptadas con umbral**: lecturas con confianza ≥ `OCR_CONF_MIN` (la app solo decide sola con esas) y cuántas de ellas están mal.

In [ ]:
from fast_plate_ocr import LicensePlateRecognizer
import cv2


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    m = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - m) / d, (c + m) / d)


def evaluate(onnx_path, plate_cfg, annotations, root, label):
    rec = LicensePlateRecognizer(onnx_model_path=onnx_path, plate_config_path=plate_cfg, device="cpu")
    df = pd.read_csv(annotations, dtype={"plate_text": str})
    out = []
    for _, r in df.iterrows():
        img = cv2.cvtColor(cv2.imread(str(Path(root) / r.image_path)), cv2.COLOR_BGR2RGB)
        pred = rec.run(img, return_confidence=True)[0]
        text = clean_text(pred.plate)
        probs = np.asarray(pred.char_probs).ravel()[:max(len(text), 1)]
        gt = normalize_plate(r.plate_text)
        out.append({"image_path": r.image_path, "plate_gt": gt, "raw": text, "pred": normalize_plate(text),
                    "conf": float(probs.min()), "cer_err": plate_distance(text, r.plate_text), "len": len(r.plate_text)})
    res = pd.DataFrame(out)
    res["correct"] = res["pred"] == res["plate_gt"]
    accepted = res["conf"] >= OCR_CONF_MIN
    k, n = int(res["correct"].sum()), len(res)
    summary = {"modelo": label, "n": n, "emr": k / n, "emr_ic95": wilson(k, n), "cer": res["cer_err"].sum() / res["len"].sum(),
               "aceptadas": float(accepted.mean()), "equivocadas_aceptadas": int((accepted & ~res["correct"]).sum())}
    return res, summary


base_test, base_summary = evaluate(BASE_ONNX, PLATE_CFG, DATA / "test_annotations.csv", DATA, "original")
base_val, _ = evaluate(BASE_ONNX, PLATE_CFG, DATA / "val_annotations.csv", DATA, "original")
print(json.dumps(base_summary, indent=2, default=float))

## 4. Afinamiento
Parte de los pesos de `cct_s_v2_global` (`--weights-path`) y entrena con *learning rate* bajo, *weight decay* pequeño, sin *label smoothing* y con *early stopping* sobre la exactitud de placa en *val*. Los aumentos son suaves (rotación ±8°, brillo, desenfoque, ruido): las placas de la garita varían en iluminación y foco más que en geometría.

In [ ]:
import albumentations as A

AUG = LOCAL / "aumentos.yaml"
A.save(A.Compose([
    A.Affine(translate_percent=(-0.02, 0.02), scale=(0.85, 1.05), rotate=(-8, 8),
             border_mode=cv2.BORDER_CONSTANT, fill=(0, 0, 0), p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.OneOf([A.GaussianBlur(sigma_limit=(0.2, 0.8), p=0.5), A.MotionBlur(blur_limit=(3, 5), p=0.5)], p=0.3),
    A.OneOf([A.GaussNoise(std_range=(0.01, 0.04), p=0.5), A.ImageCompression(quality_range=(50, 90), p=0.5)], p=0.3),
    A.ToGray(p=0.05),
]), str(AUG), data_format="yaml")

RUNS = LOCAL / "runs"
!fast-plate-ocr train \
  --model-config-file {MODEL_CFG} \
  --plate-config-file {PLATE_CFG} \
  --annotations {DATA}/train_annotations.csv \
  --val-annotations {DATA}/val_annotations.csv \
  --augmentation-path {AUG} \
  --weights-path {BASE_KERAS} \
  --epochs {EPOCHS} --batch-size {BATCH} --lr {LR} --weight-decay {WEIGHT_DECAY} \
  --label-smoothing 0.0 --early-stopping-patience {EARLY_STOPPING_PATIENCE} \
  --seed {SEED} --output-dir {RUNS}

In [ ]:
run_dir = max(RUNS.iterdir(), key=lambda p: p.stat().st_mtime)
best_keras = run_dir / "best.keras"
log = pd.read_csv(run_dir / "training_log.csv")
display(log.tail())
!fast-plate-ocr export --model {best_keras} --format onnx --plate-config-file {PLATE_CFG} --simplify
FT_ONNX = best_keras.with_suffix(".onnx")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
acc_cols = [c for c in ("acc", "val_acc") if c in log.columns]   # exactitud de placa completa
for c in acc_cols:
    ax[0].plot(log["epoch"] + 1, log[c], label=c)
ax[0].axhline(base_val["correct"].mean(), color="#9aa4b2", linestyle=":", label="original (val)")
ax[0].set_title("Exactitud de placa por época"); ax[0].legend(fontsize=8)
for c in [c for c in log.columns if c.endswith("loss")]:
    ax[1].plot(log["epoch"] + 1, log[c], label=c)
ax[1].set_title("Pérdida por época"); ax[1].legend(fontsize=8)
for a in ax: a.set_xlabel("época"); a.grid(alpha=.3)
fig.tight_layout(); fig.savefig(FT_DIR / "afinamiento_curvas.png", dpi=180); plt.show()

## 5. Antes / después en *test*
**McNemar exacta** sobre las placas que solo uno de los dos modelos lee bien: si *p* ≥ 0,05, la diferencia puede ser azar y **se mantiene el modelo original** (más simple, ya validado).

In [ ]:
ft_test, ft_summary = evaluate(FT_ONNX, PLATE_CFG, DATA / "test_annotations.csv", DATA, "afinado")
paired = base_test[["image_path", "plate_gt", "pred", "correct"]].merge(
    ft_test[["image_path", "pred", "correct"]], on="image_path", suffixes=("_orig", "_afin"))
b = int((paired.correct_orig & ~paired.correct_afin).sum())   # solo el original acierta
c = int((~paired.correct_orig & paired.correct_afin).sum())   # solo el afinado acierta
p_mcnemar = min(1.0, 2 * sum(comb(b + c, k) for k in range(min(b, c) + 1)) / 2 ** (b + c)) if b + c else 1.0

table = pd.DataFrame([base_summary, ft_summary]).set_index("modelo")
table["emr_ic95"] = table["emr_ic95"].map(lambda ci: f"[{ci[0]:.1%}, {ci[1]:.1%}]")
display(table.style.format({"emr": "{:.1%}", "cer": "{:.3f}", "aceptadas": "{:.1%}"}))
print(f"Solo el original acierta: {b} · solo el afinado acierta: {c} · McNemar p = {p_mcnemar:.3f}")
better = ft_summary["emr"] > base_summary["emr"] and p_mcnemar < 0.05 and \
    ft_summary["equivocadas_aceptadas"] <= base_summary["equivocadas_aceptadas"]
print("Decisión:", "ADOPTAR el modelo afinado" if better else "MANTENER el modelo original (la mejora no es significativa)")

if b + c:
    print("\nPlacas donde difieren:")
    display(paired[paired.correct_orig != paired.correct_afin])

result = {"particion": {s: int((pool.split == s).sum()) for s in ("train", "val", "test")},
          "placas_distintas": int(pool.plate_text.nunique()), "incluye_sin_confirmar": INCLUDE_UNCONFIRMED,
          "original": base_summary, "afinado": ft_summary, "solo_original": b, "solo_afinado": c,
          "mcnemar_p": p_mcnemar, "adoptar": bool(better),
          "hiperparametros": {"epochs": EPOCHS, "batch": BATCH, "lr": LR, "weight_decay": WEIGHT_DECAY, "seed": SEED}}
(FT_DIR / "afinamiento_resultados.json").write_text(json.dumps(result, indent=2, default=float, ensure_ascii=False))
print("Guardado:", FT_DIR / "afinamiento_resultados.json")

## 6. Guardar el modelo afinado
Se copia a Drive **siempre** (para poder revisarlo), pero solo se recomienda usarlo en la app si la decisión de la sección 5 fue *adoptar*. Para activarlo en la app, suba estos dos archivos a la carpeta `models/` del repositorio con estos nombres exactos:

| Archivo en Drive | Nombre en el repositorio |
|---|---|
| `ocr/afinamiento/ocr_afinado.onnx` | `models/ocr_afinado.onnx` |
| `ocr/afinamiento/ocr_afinado_plate_config.yaml` | `models/ocr_afinado_plate_config.yaml` |

`config/config.yaml` ya apunta a esas rutas: si los archivos existen la app usa el OCR afinado (la barra lateral dice «OCR: afinado (Ecuador)»); si no, sigue con el modelo original.

In [ ]:
shutil.copy(FT_ONNX, FT_DIR / "ocr_afinado.onnx")
shutil.copy(PLATE_CFG, FT_DIR / "ocr_afinado_plate_config.yaml")
shutil.copy(best_keras, FT_DIR / "ocr_afinado.keras")
print("Copiado a", FT_DIR, "·", "recomendado para la app" if better else "NO recomendado para la app (ver sección 5)")

## 7. Opcional: medir el OCR con los recortes del detector (garita)
La evaluación de SAVH usa recortes que ya aisló la cámara. En la garita, el recorte lo hace **YOLO11-s**. Esta sección:

1. lee `garita/imagenes/` y `garita/ground_truth.csv` (`image_path, plate_gt`; ver sección 7b del notebook de backend);
2. detecta la placa con los pesos entrenados de YOLO11-s y la recorta como la app (margen 6 %);
3. mide **EMR punta a punta** con el OCR original y con el afinado;
4. guarda los recortes en `ocr/afinamiento/garita_recortes/`, que la sección 1 usa como datos extra en una próxima corrida.

Se salta si no existe el CSV de la garita.

In [ ]:
GARITA_GT = WORK_DIR / "garita" / "ground_truth.csv"
YOLO_WEIGHTS = WORK_DIR / "experimentos" / "license-plate-recognition-rxg4e-v11" / "runs" / "YOLO11_yolo11s_completo" / "weights" / "best.pt"
if not GARITA_GT.exists():
    print(f"Se salta: no existe {GARITA_GT}.")
else:
    !pip install -q "ultralytics==8.4.177"
    from ultralytics import YOLO
    from src.data_processing import crop_with_margin
    detector = YOLO(str(YOLO_WEIGHTS))
    gt = pd.read_csv(GARITA_GT, dtype=str)
    crops_dir = FT_DIR / "garita_recortes"
    crops_dir.mkdir(exist_ok=True)
    readers = {"original": LicensePlateRecognizer(onnx_model_path=BASE_ONNX, plate_config_path=PLATE_CFG, device="cpu"),
               "afinado": LicensePlateRecognizer(onnx_model_path=FT_ONNX, plate_config_path=PLATE_CFG, device="cpu")}
    rows, crops = [], []
    for i, r in gt.iterrows():
        path = Path(r.image_path) if Path(r.image_path).is_absolute() else WORK_DIR / "garita" / r.image_path
        img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        boxes = detector.predict(img[..., ::-1], imgsz=1280, conf=0.30, verbose=False)[0].boxes
        row = {"image_path": r.image_path, "plate_gt": normalize_plate(r.plate_gt), "detectada": len(boxes) > 0}
        if len(boxes):
            crop = crop_with_margin(img, boxes.xyxy[int(boxes.conf.argmax())].tolist(), 0.06)
            crop_path = crops_dir / f"garita_{i:04d}.jpg"
            cv2.imwrite(str(crop_path), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
            crops.append({"crop_path": str(crop_path), "plate_gt": r.plate_gt})
            for name, reader in readers.items():
                row[name] = normalize_plate(reader.run(crop)[0].plate)
        rows.append(row)
    garita_res = pd.DataFrame(rows)
    pd.DataFrame(crops).to_csv(crops_dir / "recortes.csv", index=False)
    summary = {name: float((garita_res.get(name) == garita_res["plate_gt"]).mean()) for name in readers}
    summary["detectadas"] = float(garita_res["detectada"].mean())
    summary["n"] = len(garita_res)
    print("EMR punta a punta en la garita (detector + OCR):", json.dumps(summary, indent=2))
    (FT_DIR / "garita_e2e.json").write_text(json.dumps(summary, indent=2))
    display(garita_res)
    print("Nota: si estos recortes se usan como datos de entrenamiento (sección 1), la evaluación en la garita deja de ser"
          " independiente: reserve un grupo de imágenes de la garita solo para prueba.")

## 8. Copiar resultados al repositorio
| Archivo en Drive | Destino en el repositorio |
|---|---|
| `ocr/afinamiento/afinamiento_resultados.json` | `results/metrics/ocr_afinamiento.json` |
| `ocr/afinamiento/garita_e2e.json` (si existe) | `results/metrics/e2e_garita.json` |
| `ocr/afinamiento/afinamiento_curvas.png` | `results/figures/ocr_afinamiento_curvas.png` |
| `ocr_afinado.onnx` y `ocr_afinado_plate_config.yaml` (solo si la decisión fue *adoptar*) | `models/` |

**Cuidado con la interpretación:** con unas 400 placas legibles, el *test* tiene alrededor de 100 imágenes; cada acierto vale ≈1 punto de EMR. Por eso la decisión usa McNemar y no solo la diferencia de EMR.